# 04 — Accommodation and Business Capacity
## Trade Winds & Headwinds: USVI, 2000–2025

Sections:
1. USVIBER room inventory and hotel count (survey-based, annual)
2. Census guestroom crosswalk (NAICS 7211, 2007–2022)
3. Census establishment counts and receipts
4. Occupancy and utilization
5. Room inventory vs. visitor volume

**Source divergence note:** USVIBER counts all accommodation providers via annual survey. Census NAICS 7211 covers only payroll-filing establishments. Counts differ (~30% gap at comparable years). Both series are retained and labeled separately.

In [1]:
import sys, importlib
from pathlib import Path
import os

PROJ_ROOT = Path("..").resolve()
if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))
os.chdir(PROJ_ROOT)

import numpy as np
import pandas as pd
import plotly.graph_objects as go

import src.cleaning as _c
import src.charts as _ch

importlib.reload(_c)
importlib.reload(_ch)

from src.cleaning import (
    load_usviber_tour24,
    load_census_rooms,
    load_census_2007,
    load_census_2012_basic,
    load_census_2017_basic,
    load_census_2022_basic,
)
from src.charts import PALETTE, line_chart, bar_chart

pd.set_option("display.float_format", "{:,.1f}".format)
print("Ready.")

Ready.


In [2]:
tour = load_usviber_tour24()
rooms = load_census_rooms()
c07 = load_census_2007()
c12 = load_census_2012_basic()
c17 = load_census_2017_basic()
c22 = load_census_2022_basic()

# Deduplicated Census NAICS 72/721/722
frames = []
for df, v in [(c07, 2007), (c12, 2012), (c17, 2017), (c22, 2022)]:
    sub = df[df["naics"].astype(str).isin(["72", "721", "722"])][
        ["year", "naics", "naics_label", "estab", "rcptot_k", "payann_k", "emp"]
    ].copy()
    sub["vintage"] = v
    frames.append(sub)
census = (
    pd.concat(frames)
    .drop_duplicates(subset=["year", "naics"])
    .sort_values(["naics", "year"])
)

print("Census NAICS 721 (Accommodation):")
print(
    census[census["naics"] == "721"][
        ["year", "estab", "emp", "payann_k", "rcptot_k"]
    ].to_string(index=False)
)

Census NAICS 721 (Accommodation):
 year  estab     emp  payann_k  rcptot_k
 2012     57 3,876.0 111,879.0 325,265.0
 2017     51 3,385.0  99,222.0 320,666.0
 2022     57 1,650.0  55,544.0 240,810.0


---
## 1. USVIBER Room Inventory

In [3]:
fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=tour["year"],
        y=tour["hotel_rooms"],
        name="Hotel rooms",
        marker_color=PALETTE["rooms"],
    )
)
fig.add_trace(
    go.Bar(
        x=tour["year"],
        y=tour["condo_units"],
        name="Condo & other units",
        marker_color=PALETTE["cruise"],
    )
)
fig.update_layout(
    barmode="stack",
    title=dict(
        text="USVI Accommodation Inventory — USVIBER Survey<br>"
        '<span style="font-size:13px;color:#555">Annual survey. Hotels + condo/vacation rental units.</span>',
        font=dict(size=17),
    ),
    yaxis_title="Rooms or units",
    yaxis_tickformat=",",
    height=440,
    bargap=0.2,
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text='<span style="color:#888;font-size:11px">Source: USVIBER Annual Tourism Indicator 2024</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
for x_val, label in [(2017, "Irma & Maria"), (2020, "COVID-19")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )
fig.show()

In [4]:
fig = bar_chart(
    tour,
    x="year",
    y="hotel_count",
    title="Number of Hotels — USVIBER Survey",
    y_label="Number of hotels",
    color=PALETTE["rooms"],
    source="USVIBER Annual Tourism Indicator 2024",
    shock_x=[2017, 2020],
)
fig.show()
print(
    tour[["year", "hotel_count", "hotel_rooms", "condo_units", "total_rooms"]]
    .set_index("year")
    .to_string()
)

      hotel_count  hotel_rooms  condo_units  total_rooms
year                                                    
2000           49         3947         1061         5008
2010           47         3878         1131         5008
2011           47         3759         1130         4889
2012           46         3874         1157         5030
2013           44         3897         1120         5016
2014           44         3856         1118         4974
2015           41         3903         1157         5060
2016           42         3819         1023         4842
2017           41         3820         1065         4886
2018           33         2854          401         3255
2019           34         3050          463         3513
2020           23         2554          736         3291
2021           22         2367          627         2994
2022           23         2649          580         3229
2023           23         2788          612         3400
2024           24         3127 

---
## 2. Census Guestroom Crosswalk

| Vintage | Method | Rooms | Establishments |
|---|---|---|---|
| 2007 | A10 industry total (PSCODE 20000) | 4,065 | 47 |
| 2012 | A10 industry total | 5,027 | 54 |
| 2017 | NAPCS 7003825000 (room accommodation) — **Dec 31, 2017, post-hurricane** | 2,624 | 46 |
| 2022 | NAPCS 7003825000 | 2,039 | 51 |

Cross-vintage comparability is approximate; table structures differ between 2007/2012 and 2017/2022.

In [5]:
# Overlay Census rooms with USVIBER hotel rooms
fig = go.Figure()

# USVIBER hotel rooms (annual)
fig.add_trace(
    go.Scatter(
        x=tour["year"],
        y=tour["hotel_rooms"],
        mode="lines",
        name="USVIBER hotel rooms (annual survey)",
        line=dict(color=PALETTE["rooms"], width=2),
    )
)
fig.add_trace(
    go.Scatter(
        x=tour["year"],
        y=tour["total_rooms"],
        mode="lines",
        name="USVIBER total rooms (hotels + condos)",
        line=dict(color=PALETTE["total"], width=2, dash="dash"),
    )
)
# Census rooms (quinquennial snapshots)
fig.add_trace(
    go.Scatter(
        x=rooms["year"],
        y=rooms["rooms"],
        mode="markers",
        name="Census NAICS 7211 rooms (Dec 31)",
        marker=dict(color=PALETTE["cruise"], size=12, symbol="diamond"),
    )
)

fig.update_layout(
    title=dict(
        text="Room Inventory: USVIBER Survey vs. Census NAICS 7211<br>"
        '<span style="font-size:13px;color:#555">Two different measurements of accommodation capacity — different scope and methodology</span>',
        font=dict(size=17),
    ),
    yaxis_title="Rooms / units",
    yaxis_tickformat=",",
    height=460,
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text='<span style="color:#888;font-size:11px">Sources: USVIBER Annual Tourism Indicator 2024; U.S. Census Economic Census 2007–2022</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
for x_val, label in [(2017, "Irma & Maria"), (2020, "COVID-19")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )
fig.show()

print("\nCensus rooms crosswalk:")
print(
    rooms[["year", "rooms", "estab", "rcptot_k", "methodology_note"]].to_string(
        index=False
    )
)


Census rooms crosswalk:
 year  rooms  estab  rcptot_k                                                                                                                methodology_note
 2007   4065     47    308481                                                                                               A10 industry total (PSCODE 20000)
 2012   5027     54    319049                                                                                             A10 industry total (first USVI row)
 2017   2624     46    237838 NAPCS02 product 7003825000 (Room or unit accommodation for travelers). Guestroom reference date = Dec 31, 2017, post-hurricane.
 2022   2039     51    135204                                                           NAPCS02 product 7003825000 (Room or unit accommodation for travelers)


---
## 3. Census Establishments and Receipts

In [6]:
# Establishments and receipts for NAICS 72, 721, 722
fig = go.Figure()
for naics_code, name, color in [
    ("72", "NAICS 72 (Accommodation + food)", PALETTE["total"]),
    ("721", "NAICS 721 (Accommodation only)", PALETTE["tourists"]),
    ("722", "NAICS 722 (Food services)", PALETTE["cruise"]),
]:
    sub = (
        census[census["naics"] == naics_code]
        .dropna(subset=["estab"])
        .sort_values("year")
    )
    fig.add_trace(
        go.Scatter(
            x=sub["year"],
            y=sub["estab"],
            mode="markers+lines",
            name=name,
            line=dict(color=color, width=2, dash="dot"),
            marker=dict(size=10, symbol="diamond"),
        )
    )
fig.update_layout(
    title=dict(
        text="Census Establishment Counts<br>"
        '<span style="font-size:13px;color:#555">Quinquennial. Only payroll-filing establishments.</span>',
        font=dict(size=17),
    ),
    yaxis_title="Number of establishments",
    height=420,
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text='<span style="color:#888;font-size:11px">Source: U.S. Census Economic Census 2007–2022 — USVI territory level</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
for x_val, label in [(2017, "Irma & Maria"), (2020, "COVID-19")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )
fig.show()

In [7]:
# Accommodation vs food services receipts split
c_721 = census[census["naics"] == "721"][["year", "rcptot_k"]].rename(
    columns={"rcptot_k": "accom_k"}
)
c_722 = census[census["naics"] == "722"][["year", "rcptot_k"]].rename(
    columns={"rcptot_k": "food_k"}
)
rcpt = c_721.merge(c_722, on="year").dropna()
rcpt["accom_m"] = rcpt["accom_k"] / 1000
rcpt["food_m"] = rcpt["food_k"] / 1000

fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=rcpt["year"],
        y=rcpt["accom_m"],
        name="Accommodation (NAICS 721)",
        marker_color=PALETTE["tourists"],
    )
)
fig.add_trace(
    go.Bar(
        x=rcpt["year"],
        y=rcpt["food_m"],
        name="Food services (NAICS 722)",
        marker_color=PALETTE["cruise"],
    )
)
fig.update_layout(
    barmode="stack",
    title=dict(
        text="Census Receipts: Accommodation vs. Food Services (Nominal)<br>"
        '<span style="font-size:13px;color:#555">Millions of nominal USD. Quinquennial snapshots.</span>',
        font=dict(size=17),
    ),
    yaxis_title="Receipts ($M, nominal)",
    height=420,
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text='<span style="color:#888;font-size:11px">Source: U.S. Census Economic Census 2007–2022 — USVI</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
for x_val, label in [(2017, "Irma & Maria")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )
fig.show()

---
## 4. Occupancy and Utilization

In [8]:
# Occupancy rate alongside available rooms
fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=tour["year"],
        y=tour["total_rooms"],
        name="Available rooms/units",
        marker_color="#c8e6f7",
        yaxis="y",
    )
)
fig.add_trace(
    go.Scatter(
        x=tour["year"],
        y=tour["occupancy_rate_pct"],
        mode="lines+markers",
        name="Occupancy rate (%)",
        line=dict(color=PALETTE["occupancy"], width=2.5),
        yaxis="y2",
    )
)
fig.update_layout(
    title=dict(
        text="Room Inventory and Occupancy Rate<br>"
        '<span style="font-size:13px;color:#555">Bars = available rooms (left axis); line = occupancy % (right axis)</span>',
        font=dict(size=17),
    ),
    yaxis=dict(title="Available rooms/units", tickformat=","),
    yaxis2=dict(
        title="Occupancy rate (%)",
        overlaying="y",
        side="right",
        range=[0, 100],
        showgrid=False,
    ),
    height=460,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text='<span style="color:#888;font-size:11px">Source: USVIBER Annual Tourism Indicator 2024</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
for x_val, label in [(2017, "Irma & Maria"), (2020, "COVID-19")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )
fig.show()

---
## 5. Room Inventory vs. Visitor Volume

In [9]:
# Tourists per available room — rough utilization indicator
# (tourists in thousands, rooms in units — multiply tourists by 1000 first)
tour2 = tour.copy()
tour2["tourists_per_room"] = (tour2["tourists"] * 1000) / tour2["total_rooms"]

fig = bar_chart(
    tour2,
    x="year",
    y="tourists_per_room",
    title="Annual Overnight Tourists per Available Room",
    subtitle="Rough utilization indicator. Assumes constant room availability across the year.",
    y_label="Annual tourists per room",
    color=PALETTE["tourists"],
    source="USVIBER Annual Tourism Indicator 2024 — author calculations",
    shock_x=[2017, 2020],
)
fig.show()
print(
    tour2[["year", "tourists", "total_rooms", "tourists_per_room"]]
    .set_index("year")
    .round(1)
    .to_string()
)

      tourists  total_rooms  tourists_per_room
year                                          
2000     546.7         5008              109.2
2010     576.2         5008              115.1
2011     566.9         4889              116.0
2012     619.2         5030              123.1
2013     590.3         5016              117.7
2014     614.8         4974              123.6
2015     641.9         5060              126.9
2016     667.9         4842              137.9
2017     535.0         4886              109.5
2018     380.9         3255              117.0
2019     418.0         3513              119.0
2020     135.6         3291               41.2
2021     396.2         2994              132.3
2022     402.6         3229              124.7
2023     378.0         3400              111.2
2024     380.3         3735              101.8
